In [23]:
import torch
from transformers import ViTImageProcessorFast, ViTForImageClassification, BitsAndBytesConfig


# 配置4-bit量化
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,  # 启用4bit加载
    bnb_4bit_compute_dtype=torch.bfloat16,  # 计算时使用的数据类型
    bnb_4bit_quant_type="nf4",  # 量化类型：nf4或fp4
    bnb_4bit_use_double_quant=True,  # 是否使用双重量化
)


image_processor = ViTImageProcessorFast.from_pretrained('F:/03Models/vit-base-patch16-224')

model = ViTForImageClassification.from_pretrained(
    'F:/03Models/vit-base-patch16-224',
    device_map="auto",
    quantization_config=bnb_config
)


In [24]:
# 处理流程  - 预处理
img_file = "./imgs/000000039769.jpg"
# image = Image.open(img_file)
inputs = image_processor(images=img_file, return_tensors="pt").to("cuda")   # 把模型输入也移动到GPU
pixel_values = inputs.pixel_values

# 处理流程  - 推理
with torch.no_grad():
    # outputs = classfication(pixel_values)
    outputs = model(**inputs)
logits = outputs.logits
logits.shape   # 1000个类别

# 处理流程  - 后处理
prediction = logits.argmax(-1)
# print(prediction)  # 预测结果

# 转换为可读标签
label = model.config.id2label[prediction.item()]
print("分类结果：", label)

分类结果： Egyptian cat


In [28]:
print(type(model.vit.embeddings.patch_embeddings.projection))
print(model.vit.embeddings.patch_embeddings.projection.weight.dtype)

<class 'torch.nn.modules.conv.Conv2d'>
torch.float16


In [6]:
import torch
import torch.ao.quantization as tq
import torch.nn as nn
import torch.nn.intrinsic as nni
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification,
    BertConfig,
    BertForSequenceClassification
)
from torch.utils.data import DataLoader, Dataset
import numpy as np
from tqdm import tqdm
import os
import time
import copy
from typing import Dict, List, Optional, Tuple
import warnings
warnings.filterwarnings('ignore')

# 设置随机种子
torch.manual_seed(42)
np.random.seed(42)

class TextClassificationDataset(Dataset):
    """文本分类数据集"""
    def __init__(self, texts: List[str], labels: List[int], tokenizer, max_length: int = 128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self) -> int:
        return len(self.texts)
    
    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        text = str(self.texts[idx])
        label = self.labels[idx]
        
        # 编码文本
        encoding = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )
        
        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'labels': torch.tensor(label, dtype=torch.long)
        }

class BertForQuantization(BertForSequenceClassification):
    """支持量化的BERT模型类"""
    def __init__(self, config):
        super().__init__(config)
        self.quant = tq.QuantStub()
        self.quantized = False
        self.skip_quantization = ['embeddings', 'pooler']  # 跳过量化层
        self.dequant = tq.DeQuantStub()
    
    def forward(self, input_ids=None, attention_mask=None, token_type_ids=None, **kwargs):
        # 标准forward方法
        input_ids = self.quant(input_ids)
        attention_mask = self.quant(attention_mask)
        token_type_ids = self.quant(token_type_ids)
        x = super().forward(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
            **kwargs
        )
        x = self.dequant(x)
        return x
    
    def fuse_model(self):
        """融合BERT中的层以优化量化"""
        # 注意：BERT的量化融合需要针对具体架构进行
        # 这里提供一个基本的融合示例
        
        # 融合注意力层中的线性层
        for layer_idx in range(self.config.num_hidden_layers):
            attention_layer = self.bert.encoder.layer[layer_idx].attention
            
            # 融合query、key、value的线性层
            nni.fuse_modules(
                attention_layer.self,
                [['query', 'key', 'value']],
                inplace=True
            )
            
            # 融合输出层
            nni.fuse_modules(
                attention_layer.output,
                [['dense', 'LayerNorm']],
                inplace=True
            )
            
            # 融合FFN层
            ffn_layer = self.bert.encoder.layer[layer_idx].intermediate
            output_layer = self.bert.encoder.layer[layer_idx].output
            
            nni.fuse_modules(
                ffn_layer,
                [['dense', 'intermediate_act_fn']],
                inplace=True
            )
        
        print("模型层融合完成")

def prepare_data() -> Tuple[DataLoader, DataLoader, DataLoader]:
    """准备训练、校准和测试数据"""
    
    # 生成更多样化的示例数据
    texts = [
        # 正面评价
        "This movie is absolutely fantastic! Great acting and storyline.",
        "I loved every minute of it. Highly recommended!",
        "Excellent performance by the entire cast.",
        "A masterpiece of modern cinema.",
        "Brilliant direction and stunning visuals.",
        "One of the best films I've ever seen.",
        "Truly moving and emotionally powerful.",
        "Outstanding achievement in filmmaking.",
        "Perfect blend of humor and drama.",
        "A must-watch for everyone.",
        
        # 负面评价
        "Terrible movie, complete waste of time.",
        "Poor acting and boring plot.",
        "Disappointing and predictable.",
        "Worst film of the year.",
        "Awful dialogue and bad direction.",
        "Completely unoriginal and uninspired.",
        "Boring from start to finish.",
        "Poorly written and badly executed.",
        "Forgettable and mediocre.",
        "Don't waste your money on this.",
        
        # 中性/混合评价
        "Average movie, nothing special.",
        "Some good parts but overall mediocre.",
        "Decent but forgettable.",
        "Mixed feelings about this one.",
        "Okay for a casual watch.",
        "Not great but not terrible either.",
        "Has its moments but falls short.",
        "Could have been better.",
        "Neither good nor bad.",
        "Just an average film.",
    ] * 5  # 重复以创建更多数据
    
    labels = []
    for i in range(5):
        # 正面: 0-9, 负面: 10-19, 中性: 20-29
        labels.extend([2] * 10 + [0] * 10 + [1] * 10)  # 2:正面, 0:负面, 1:中性
    
    # 加载分词器
    tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
    
    # 创建数据集
    full_dataset = TextClassificationDataset(texts, labels, tokenizer)
    
    # 分割数据集
    train_size = int(0.6 * len(full_dataset))
    calib_size = int(0.2 * len(full_dataset))
    test_size = len(full_dataset) - train_size - calib_size
    
    train_dataset, calib_dataset, test_dataset = torch.utils.data.random_split(
        full_dataset, [train_size, calib_size, test_size]
    )
    
    # 创建数据加载器
    train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
    calib_loader = DataLoader(calib_dataset, batch_size=8, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False)
    
    print(f"训练集大小: {len(train_dataset)}")
    print(f"校准集大小: {len(calib_dataset)}")
    print(f"测试集大小: {len(test_dataset)}")
    
    return train_loader, calib_loader, test_loader, tokenizer

def evaluate_model(model, test_loader: DataLoader, device: torch.device) -> Tuple[float, float]:
    """评估模型性能"""
    model.eval()
    correct = 0
    total = 0
    total_time = 0
    
    with torch.no_grad():
        for batch in tqdm(test_loader, desc="评估中"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            # 测量推理时间
            if device.type == 'cpu':
                torch.cuda.synchronize() if device.type == 'cuda' else None
                start_time = time.time()
                outputs = model(input_ids=input_ids, attention_mask=attention_mask)
                end_time = time.time()
                total_time += (end_time - start_time)
            else:
                start_event = torch.cuda.Event(enable_timing=True)
                end_event = torch.cuda.Event(enable_timing=True)
                start_event.record()
                outputs = model(input_ids=input_ids, attention_mask=attention_mask)
                end_event.record()
                torch.cuda.synchronize()
                total_time += start_event.elapsed_time(end_event) / 1000  # 转换为秒
            
            predictions = torch.argmax(outputs.logits, dim=-1)
            total += labels.size(0)
            correct += (predictions == labels).sum().item()
    
    accuracy = 100 * correct / total
    avg_time = total_time / len(test_loader)
    
    return accuracy, avg_time

def static_quantization_with_ao(model: nn.Module, 
                                calib_loader: DataLoader,
                                qconfig_spec: Optional[Dict] = None) -> nn.Module:
    """
    使用torch.ao进行静态量化
    """
    print("\n=== 开始静态量化 ===")
    
    # 复制模型以避免修改原始模型
    model_to_quantize = copy.deepcopy(model)
    model_to_quantize.eval()
    
    # 1. 设置量化配置
    if qconfig_spec is None:
        # 使用默认的x86量化配置
        model_to_quantize.qconfig = tq.get_default_qconfig('x86')
    else:
        model_to_quantize.qconfig = qconfig_spec
    model_to_quantize.qconfig = tq.float_qparams_weight_only_qconfig
    # 2. 准备模型用于量化
    print("准备模型用于量化...")
    model_prepared = tq.prepare(model_to_quantize, inplace=False)
    
    # 3. 使用校准数据进行校准
    print("使用校准数据进行校准...")
    with torch.no_grad():
        for batch in tqdm(calib_loader, desc="校准"):
            input_ids = batch['input_ids']
            attention_mask = batch['attention_mask']
            _ = model_prepared(input_ids=input_ids, attention_mask=attention_mask)
    
    # 4. 转换为量化模型
    print("转换为量化模型...")
    model_quantized = tq.convert(model_prepared, inplace=False)
    
    # 标记为量化模型
    model_quantized.quantized = True
    
    print("量化完成！")
    return model_quantized

def custom_quantization_config() -> Dict:
    """自定义量化配置"""
    
    # 创建自定义的观察器配置
    my_qconfig = tq.QConfig(
        activation=tq.MinMaxObserver.with_args(
            dtype=torch.quint8,
            qscheme=torch.per_tensor_affine,
            reduce_range=False
        ),
        weight=tq.PerChannelMinMaxObserver.with_args(
            dtype=torch.qint8,
            qscheme=torch.per_channel_symmetric
        )
    )
    
    # 也可以使用不同的后端配置
    qconfigs = {
        'x86': tq.get_default_qconfig('x86'),      # 适合服务器CPU
        'qnnpack': tq.get_default_qconfig('qnnpack'),  # 适合移动设备
        'fbgemm': tq.get_default_qconfig('fbgemm'),    # 适合服务器（FBGEMM）
        'custom': my_qconfig
    }
    
    return qconfigs

def measure_model_size(model: nn.Module, name: str = "model") -> float:
    """测量模型大小"""
    temp_path = f"temp_{name}.pth"
    torch.save(model.state_dict(), temp_path)
    size_mb = os.path.getsize(temp_path) / (1024 * 1024)
    os.remove(temp_path)
    return size_mb

def compare_models(original_model: nn.Module, 
                   quantized_model: nn.Module,
                   test_loader: DataLoader,
                   device: torch.device):
    """比较原始模型和量化模型的性能"""
    
    print("\n" + "="*60)
    print("模型性能对比")
    print("="*60)
    
    # 评估原始模型
    print("\n📊 原始模型评估:")
    original_acc, original_time = evaluate_model(original_model, test_loader, device)
    original_size = measure_model_size(original_model, "original")
    print(f"   准确率: {original_acc:.2f}%")
    print(f"   推理时间: {original_time*1000:.2f} ms/批")
    print(f"   模型大小: {original_size:.2f} MB")
    
    # 将量化模型移到CPU（量化模型通常在CPU上运行）
    quantized_model = quantized_model.to('cpu')
    
    # 评估量化模型
    print("\n📊 量化模型评估:")
    quantized_acc, quantized_time = evaluate_model(quantized_model, test_loader, torch.device('cpu'))
    quantized_size = measure_model_size(quantized_model, "quantized")
    print(f"   准确率: {quantized_acc:.2f}%")
    print(f"   推理时间: {quantized_time*1000:.2f} ms/批")
    print(f"   模型大小: {quantized_size:.2f} MB")
    
    # 计算改进
    print("\n📈 改进统计:")
    print(f"   准确率变化: {quantized_acc - original_acc:+.2f}%")
    print(f"   推理速度提升: {(original_time/quantized_time):.2f}x")
    print(f"   模型压缩比: {original_size/quantized_size:.2f}x")
    
    return {
        'original': {'accuracy': original_acc, 'time': original_time, 'size': original_size},
        'quantized': {'accuracy': quantized_acc, 'time': quantized_time, 'size': quantized_size}
    }

def save_quantized_model(model: nn.Module, tokenizer, path: str):
    """保存量化模型和分词器"""
    # 保存模型
    torch.save({
        'model_state_dict': model.state_dict(),
        'quantized': getattr(model, 'quantized', False),
        'config': model.config
    }, path)
    
    # 保存分词器
    tokenizer.save_pretrained(path.replace('.pt', '_tokenizer'))
    
    print(f"量化模型已保存到: {path}")
    print(f"分词器已保存到: {path.replace('.pt', '_tokenizer')}")

def load_quantized_model(path: str, device: torch.device) -> nn.Module:
    """加载量化模型"""
    checkpoint = torch.load(path, map_location=device)
    
    # 重新创建模型
    config = checkpoint['config']
    model = BertForQuantization.from_pretrained(
        'bert-base-uncased',
        config=config
    )
    
    # 加载权重
    model.load_state_dict(checkpoint['model_state_dict'])
    model.quantized = checkpoint.get('quantized', False)
    
    return model

def inference_example(model: nn.Module, tokenizer, texts: List[str], device: torch.device):
    """推理示例"""
    model.eval()
    
    print("\n" + "="*60)
    print("推理示例")
    print("="*60)
    
    # 情感标签映射
    id2label = {0: "负面 ⭐", 1: "中性 ⭐⭐", 2: "正面 ⭐⭐⭐"}
    
    with torch.no_grad():
        for text in texts:
            # 编码输入
            inputs = tokenizer(
                text, 
                return_tensors='pt', 
                padding=True, 
                truncation=True,
                max_length=128
            )
            
            # 移动到设备
            inputs = {k: v.to(device) for k, v in inputs.items()}
            
            # 推理
            start_time = time.time()
            outputs = model(**inputs)
            inference_time = (time.time() - start_time) * 1000  # 转换为毫秒
            
            # 获取预测
            predictions = torch.softmax(outputs.logits, dim=-1)
            predicted_class = torch.argmax(predictions, dim=-1).item()
            confidence = predictions[0][predicted_class].item()
            
            print(f"\n📝 文本: {text}")
            print(f"   🏷️ 预测: {id2label[predicted_class]}")
            print(f"   📊 置信度: {confidence:.2%}")
            print(f"   ⏱️ 推理时间: {inference_time:.2f} ms")

def main():
    """主函数"""
    print("="*60)
    print("静态量化Transformers模型示例 (使用torch.ao)")
    print("="*60)
    
    # 设置设备
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"\n使用设备: {device}")
    
    # 1. 准备数据
    print("\n📁 准备数据...")
    train_loader, calib_loader, test_loader, tokenizer = prepare_data()
    
    # 2. 加载预训练模型
    print("\n🤖 加载预训练模型...")
    model_name = 'bert-base-uncased'
    num_labels = 3  # 正面、负面、中性
    
    original_model = BertForQuantization.from_pretrained(
        model_name,
        num_labels=num_labels
    )
    original_model.to(device)
    
    print(f"模型架构: {model_name}")
    print(f"分类数量: {num_labels}")
    
    # 3. 评估原始模型
    print("\n📊 评估原始模型...")
    original_model.eval()
    
    # 4. 尝试不同的量化配置
    print("\n🔧 尝试不同的量化配置...")
    qconfigs = custom_quantization_config()
    
    results = {}
    
    for qconfig_name, qconfig in qconfigs.items():
        print(f"\n{'='*40}")
        print(f"使用量化配置: {qconfig_name}")
        print(f"{'='*40}")
        
        # 静态量化
        quantized_model = static_quantization_with_ao(
            original_model.cpu(),  # 量化通常在CPU上进行
            calib_loader,
            qconfig_spec=qconfig
        )
        
        # 比较性能
        comparison = compare_models(
            original_model.cpu(),
            quantized_model,
            test_loader,
            torch.device('cpu')
        )
        
        results[qconfig_name] = comparison
    
    # 5. 选择最佳模型（基于准确率）
    print("\n" + "="*60)
    print("🏆 最佳量化配置")
    print("="*60)
    
    best_config = None
    best_acc = -float('inf')
    
    for config_name, comparison in results.items():
        quantized_acc = comparison['quantized']['accuracy']
        if quantized_acc > best_acc:
            best_acc = quantized_acc
            best_config = config_name
    
    print(f"最佳量化配置: {best_config}")
    print(f"最佳量化准确率: {best_acc:.2f}%")
    
    # 6. 使用最佳配置重新量化模型
    print("\n🔄 使用最佳配置重新量化模型...")
    best_qconfig = qconfigs[best_config]
    final_quantized_model = static_quantization_with_ao(
        original_model.cpu(),
        calib_loader,
        # qconfig_spec=best_qconfig
        qconfig_spec=float_qparams_weight_only_qconfig
    )
    
    # 7. 保存量化模型
    print("\n💾 保存量化模型...")
    save_quantized_model(
        final_quantized_model, 
        tokenizer, 
        'quantized_bert_ao.pt'
    )
    
    # 8. 推理示例
    print("\n🔍 运行推理示例...")
    sample_texts = [
        "This movie is absolutely wonderful! I loved it.",
        "Terrible film, complete waste of time.",
        "It's an average movie, nothing special.",
        "Amazing performance by the lead actor!",
        "Disappointing and predictable plot.",
    ]
    
    inference_example(
        final_quantized_model, 
        tokenizer, 
        sample_texts,
        torch.device('cpu')
    )
    
    print("\n✅ 静态量化完成！")

# 高级量化技术示例
class AdvancedQuantizationTechniques:
    """高级量化技术示例"""
    
    @staticmethod
    def quantize_with_observer_selection(model: nn.Module, 
                                          calib_loader: DataLoader,
                                          observer_type: str = 'minmax'):
        """使用不同的观察器进行量化"""
        
        if observer_type == 'minmax':
            obs = tq.MinMaxObserver
        elif observer_type == 'moving_average':
            obs = tq.MovingAverageMinMaxObserver
        elif observer_type == 'histogram':
            obs = tq.HistogramObserver
        elif observer_type == 'per_channel':
            obs = tq.PerChannelMinMaxObserver
        else:
            raise ValueError(f"Unknown observer type: {observer_type}")
        
        qconfig = tq.QConfig(
            activation=obs.with_args(
                dtype=torch.quint8,
                qscheme=torch.per_tensor_affine
            ),
            weight=obs.with_args(
                dtype=torch.qint8,
                qscheme=torch.per_tensor_symmetric
            )
        )
        
        # model.qconfig = qconfig
        model.qconfig = tq.float_qparams_weight_only_qconfig
        model_prepared = tq.prepare(model, inplace=False)
        
        with torch.no_grad():
            for batch in calib_loader:
                _ = model_prepared(batch['input_ids'], batch['attention_mask'])
        
        return tq.convert(model_prepared, inplace=False)
    
    @staticmethod
    def quantize_with_dtype_selection(model: nn.Module,
                                       calib_loader: DataLoader,
                                       activation_dtype=torch.quint8,
                                       weight_dtype=torch.qint8):
        """选择不同的数据类型进行量化"""
        
        qconfig = tq.QConfig(
            activation=tq.MinMaxObserver.with_args(dtype=activation_dtype),
            weight=tq.MinMaxObserver.with_args(dtype=weight_dtype)
        )
        
        model.qconfig = qconfig
        model_prepared = tq.prepare(model, inplace=False)
        
        with torch.no_grad():
            for batch in calib_loader:
                _ = model_prepared(batch['input_ids'], batch['attention_mask'])
        
        return tq.convert(model_prepared, inplace=False)

# 添加模型验证功能
def validate_quantized_model(quantized_model: nn.Module,
                             test_loader: DataLoader,
                             num_samples: int = 100):
    """验证量化模型的输出"""
    
    print("\n" + "="*60)
    print("量化模型验证")
    print("="*60)
    
    quantized_model.eval()
    sample_count = 0
    
    with torch.no_grad():
        for batch in test_loader:
            if sample_count >= num_samples:
                break
            
            input_ids = batch['input_ids'][:1]  # 只取第一个样本
            attention_mask = batch['attention_mask'][:1]
            
            # 检查输出类型
            output = quantized_model(input_ids=input_ids, attention_mask=attention_mask)
            
            if sample_count == 0:
                print(f"输入数据类型: {input_ids.dtype}")
                print(f"输出数据类型: {output.logits.dtype}")
                print(f"模型参数量化状态: {any(isinstance(p, torch.quantized.QTensor) for p in quantized_model.parameters())}")
            
            sample_count += 1
    
    print(f"验证完成，检查了 {sample_count} 个样本")


main()

静态量化Transformers模型示例 (使用torch.ao)

使用设备: cuda

📁 准备数据...
训练集大小: 90
校准集大小: 30
测试集大小: 30

🤖 加载预训练模型...


Some weights of BertForQuantization were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


模型架构: bert-base-uncased
分类数量: 3

📊 评估原始模型...

🔧 尝试不同的量化配置...

使用量化配置: x86

=== 开始静态量化 ===
准备模型用于量化...
使用校准数据进行校准...


校准: 100%|██████████████████████████████████████████████████████████████████████████████| 4/4 [00:01<00:00,  2.82it/s]


转换为量化模型...


Exception: calculate_qparams should not be called for PlaceholderObserver